# PT-W4-D3 实验：四物种 Rule Card 引擎 —— 让「费用未清算不许退场」变成 AI 可推理

配套阅读：`PT-W4-D3-补RuleModel.md`。

**实验目标**：把 BCM 业务规则列里的散文（如 CRE-LEA-011「费用未清算不许退场」）升级为结构化 **Rule Card**（ID + 物种 + 谓词 + 挂载点 + 证据 + 变体），并用 60 行代码跑通：

1. **L3 规则判断**：A101 铺位「能不能出租」由 Derivation 规则推导，失败谓词即「原因」；
2. **Guard 评估**：终止守卫（Inspection ∧ 清算）逐谓词出解释；
3. **Variant 切换**：万达版 / 中旅版进场守卫，同一 Rule ID 按租户换谓词；
4. **Invariant 断言**：双 Active Occupancy 触发系统级错误报告。

注意：谓词引用的全部是 **D1 的 Entity/Relationship、D2 的 Lifecycle 概念**（不是代码路径）——这就是「规则是 Ontology 的公民」的含义。

## 1. 语义事实层（来自 D1 Relationship + D2 Lifecycle）

规则不直接读数据库，读的是**语义视图**：Entity 状态 + D1 `occupies` 关系。

In [ ]:
from dataclasses import dataclass

@dataclass
class ResourceUnit:            # D1 Entity：资源单元（物理身份归 Asset Foundation）
    code: str
    operational_status: str    # D2 状态机：active / inactive / under_renovation / blocked
    restriction: bool          # 是否存在限制（预定期 / 装修锁定 / 已退出等）

@dataclass
class Contract:                # D1 Entity：合同（生命周期归 Contract Lifecycle 域）
    cid: int
    status: str                # D2 状态机：draft/active/expiring/terminating/terminated/voided
    inspection_done: bool      # 退租验收是否完成
    settlement_status: str     # 清算状态：pending / in_progress / completed
    first_bill_paid: bool = True   # 首期账款是否缴清（万达进场守卫用）
    meter_installed: bool = True   # 水电表是否已挂表（中旅开业守卫用）

# D1 Relationship 实例层：Contract --occupies--> ResourceUnit（运行时沿关系解析，规则层不写死 id）
facts = {
    "spaces":    {"A101": ResourceUnit("A101", "active", restriction=False)},
    "contracts": {1234: Contract(1234, "terminating", inspection_done=False,
                                  settlement_status="in_progress")},
    "occupies":  {1234: "A101"},
}

def active_occupancies(space_code, f):
    """D1 occupies 关系 + D2 状态机联合推导：该铺位当前所有『未结束』的占用合同"""
    return [c for cid, c in f["contracts"].items()
            if f["occupies"].get(cid) == space_code
            and c.status not in ("terminated", "voided")]

print("A101 当前未结束占用合同:", [c.cid for c in active_occupancies("A101", facts)])

## 2. Rule Card：规则的 Ontology 身份

四个字段让散文变成 AI 可推理：**结构化谓词**（合取的原子谓词，每个引用 Ontology 概念）、**物种**（invariant/guard/derivation/variant，决定执行位置）、**挂载点**（挂在 D2 哪次迁移 / 写路径 / 查询层）、**证据**（回答「依据」时引用业务条文，不是代码）。

In [ ]:
@dataclass
class RuleCard:
    rule_id: str
    species: str          # invariant / guard / derivation / variant
    statement: str        # 业务语言声明（人读）
    predicates: dict      # 谓词名(Ontology概念式命名) -> callable(facts)->bool，合取
    on_violation: str     # 违反后果：Guard=拒绝且世界不变 / Invariant=系统级错误 / Derivation=结论为False
    evidence: str         # BCM 行 ID / Domain Model 章节 —— Agent 回答「依据」的出处
    mount: str            # 挂载点

def evaluate(rule, f):
    """合取评估 + 逐谓词明细。可解释性不在结论里，在 trace 里。"""
    trace = {name: bool(pred(f)) for name, pred in rule.predicates.items()}
    return all(trace.values()), trace

def report(rule, f):
    ok, trace = evaluate(rule, f)
    print(f"◆ {rule.rule_id} [{rule.species}] {rule.statement}")
    for name, v in trace.items():
        print(f"    {'✓' if v else '✗'} {name}")
    print(f"    => {'满足' if ok else '不满足'} | 违反后果: {rule.on_violation}")
    print(f"    依据: {rule.evidence} | 挂载: {rule.mount}\n")
    return ok, trace

In [ ]:
# ---- A101 场景最小规则集（md 第五节 Rule 清单的可执行版）----

R001 = RuleCard("CRE-R-001", "invariant",
    "一个 Resource Unit 同一时间只有一个 Active Occupancy",
    {"count(active occupancy of space) <= 1": lambda f: len(active_occupancies("A101", f)) <= 1},
    "系统级错误：数据已脏，需人工介入（拒绝操作救不了）",
    "MI Domain Model Lease/Occupancy 不变量 1", "写路径出口断言")

R002 = RuleCard("CRE-R-002", "derivation",
    "AvailableForLeasing = Asset.Active ∧ no Occupancy ∧ no Restriction",
    {"Asset.operational_status == active":   lambda f: f["spaces"]["A101"].operational_status == "active",
     "Occupancy.current == none":            lambda f: len(active_occupancies("A101", f)) == 0,
     "Restriction.exists == false":          lambda f: not f["spaces"]["A101"].restriction},
    "结论为 False（推导规则返回结论，不拒绝操作）",
    "MI Domain Model Lease/Occupancy 领域规则（D-001 Amendment A：可出租不是字段）",
    "查询/推理层")

R003 = RuleCard("CRE-R-003", "guard",
    "终止守卫：退租 Inspection 完成 ∧ 清算完成（不满足则 Space 不可释放、不可新签约）",
    {"Contract.inspection_status == completed": lambda f: f["contracts"][1234].inspection_done,
     "Contract.settlement_status == completed": lambda f: f["contracts"][1234].settlement_status == "completed"},
    "拒绝退场/新签约，世界保持：Contract=Terminating, Space=Occupied",
    "D2 迁移声明 Active→Terminated + CRE BCM CRE-LEA-011", "D2 迁移 Active→Terminated 入口")

print("规则注册完成：", R001.rule_id, R002.rule_id, R003.rule_id)

## 3. L3 规则判断：A101 为什么不能出租？

推理链 = D1 定位 → D2 状态 → **今天的 Rule 判断**。Derivation 失败谓词给出「原因」，Guard 告诉你「卡在哪一步」，两者合成 Agent 的回答。

In [ ]:
# Step 1: Derivation —— 铺位可出租吗？
ok_leasable, trace = report(R002, facts)

# Step 2: Guard —— 退租流程卡在哪？
ok_term, trace_g = report(R003, facts)

# Step 3: 合成 Agent 回答（D7 Digital Employee Validation 的目标输出形态）
reason = "、".join(n for n, v in trace.items() if not v)
blocked = "、".join(n for n, v in trace_g.items() if not v)
print("━━━ Agent 回答 ━━━")
print(f"问：A101 铺位为什么不能出租？")
print(f"答：不可出租。推导规则 {R002.rule_id} 失败于【{reason}】。")
print(f"    深入原因：合同 #1234 处于终止流程（Terminating），守卫 {R003.rule_id} 未满足【{blocked}】。")
print(f"    依据：{R002.evidence}；{R003.evidence}。")
print(f"    建议动作：创建 Inspection Task，完成后清算结清 → 触发 ContractTerminated")
print(f"              → occupancy-effect → A101 释放（引用 D2 Event/Effect 声明）。")

## 4. Variant：同一 Rule ID，按租户换谓词

CRE-LEA-011 的客户差异（万达「首期欠缴不许进场」/ 中旅「未挂表不许开业」）在 BCM 散文里是备注，在 Rule Model 里是**租户配置**：

In [ ]:
R020_VARIANTS = {   # CRE-R-020 进场守卫：variant 是规则的客户维度，非独立规则
    "wanda": ("万达版", {"首期账款已缴清": lambda f: f["contracts"][1234].first_bill_paid}),
    "ctg":   ("中旅版", {"水电表已挂表":   lambda f: f["contracts"][1234].meter_installed}),
}

def variant_rule(tenant):
    name, preds = R020_VARIANTS[tenant]
    return RuleCard("CRE-R-020", "variant", f"进场守卫（{name}）",
                    preds, "拒绝进场登记，世界不变",
                    "CRE BCM CRE-LEA-011 客户差异", "进场登记单入口（05 运营）")

# 同一个租户实例：首期未缴、未挂表 —— 两个租户各自被自己的守卫拦下
facts["contracts"][1234].first_bill_paid = False
facts["contracts"][1234].meter_installed = False
for tenant in ("wanda", "ctg"):
    ok, _ = report(variant_rule(tenant), facts)

# 中旅补挂表后 → 守卫通过（谓词可独立演化，不影响万达版）
facts["contracts"][1234].meter_installed = True
ok, _ = report(variant_rule("ctg"), facts)

## 5. Invariant：为什么它不是 Guard？

Guard 违反 = 拒绝这次操作，世界不变；Invariant 违反 = **世界已经脏了**。给 A101 再塞一个 Active 合同，看出口断言如何报警：

In [ ]:
# 模拟写路径 bug：绕过 Guard 直接写入第二个 Active 占用
facts["contracts"][5678] = Contract(5678, "active", inspection_done=False,
                                    settlement_status="pending")
facts["occupies"][5678] = "A101"

ok_inv, trace_inv = evaluate(R001, facts)
occ = [c.cid for c in active_occupancies("A101", facts)]
print(f"A101 未结束占用合同: {occ}（预期 1 个，实际 {len(occ)} 个）")
print(f"{R001.rule_id} [{R001.species}] => {'✓ 一致' if ok_inv else '✗✗ 违反：' + R001.on_violation}")

# 顺带验证：此时 Derivation 的失败谓词不变（no Occupancy 仍失败）——规则之间互不干扰
_, t = evaluate(R002, facts)
print(f"{R002.rule_id} 失败谓词不变: {[n for n, v in t.items() if not v]}")

# 清理，恢复场景
del facts["contracts"][5678]; del facts["occupies"][5678]
print("清理完成，A101 占用恢复:", [c.cid for c in active_occupancies("A101", facts)])

## 6. 实验结论

| 观察 | 对应 md 的论点 |
|---|---|
Derivation 失败谓词列表 = 「为什么不能出租」的**原因**，逐谓词明细 = 可解释性 | 推导规则让 Agent 从事实重算，不信任过期快照（D-001 A） |
Guard 未满足时世界保持 Terminating/Occupied 不变 | Guard 是入口检查；`ErrInvalidTransition` 原子失败同构 |
Variant 同一 Rule ID 换谓词，万达/中旅互不影响 | 客户差异从散文备注变成租户配置 |
Invariant 违反时报「系统级错误」而非「拒绝操作」 | Guard vs Invariant 的分界 = 违反时世界处于什么状态 |
回答里引用的是 Rule ID + evidence，而非代码路径 | 规则是 Ontology 的公民：每条规则像 effect_type 一样可注册、可冻结、可追溯 |